# Fine-tune YOLO26n for Cosmo's stop-sign detector

Runs the whole pipeline in the background on a Kaggle GPU: get training data → fine-tune → export for the Pi → re-run the `stop-sign-eval` tests (pretrained vs fine-tuned).

**Before running** (see `train/README.md` in the repo):
1. Settings (right panel): **Accelerator → GPU T4 x2**, **Internet → On**.
2. **+ Add Input** → attach one or more sign datasets (LISA traffic signs, road-sign-detection, a Roboflow export...). The prep step finds them automatically.
3. **Save Version → Save & Run All (Commit)**. You can close your laptop; results appear under this version's **Output** tab.

In [ ]:
# ---- settings -------------------------------------------------------------
REPO_URL = "https://github.com/ECE49022Team22/stop-sign-eval"
EPOCHS = 60              # stops early if validation stops improving
IMGSZ = 640
BATCH = 32
OPEN_IMAGES_MAX = 800    # extra stop-sign photos from Google Open Images (0 = skip)
OUT = "/kaggle/working/stop_sign_finetune"   # everything you want to keep goes here
WORK = "/tmp/work"                           # scratch (not saved)

In [ ]:
import os
!nvidia-smi -L || echo "No GPU found: set Accelerator -> GPU T4 x2 in the right panel"
os.makedirs(WORK, exist_ok=True)
REPO = f"{WORK}/stop-sign-eval"
if not os.path.exists(REPO):
    !git clone -q --depth 1 {REPO_URL} {REPO}
!pip install -q "ultralytics>=8.4"
%cd {REPO}
!git log --oneline -1

In [ ]:
# Attached datasets (from + Add Input)
!ls -la /kaggle/input 2>/dev/null || echo "No datasets attached"

In [ ]:
# Evaluation photos + controlled test set (also used to keep look-alikes out of training)
!python get_data.py
!python make_synthetic.py

In [ ]:
# Optional: extra stop-sign photos from Google Open Images (CC BY)
if OPEN_IMAGES_MAX > 0:
    !python train/fetch_open_images.py --out {WORK}/open_images --max-images {OPEN_IMAGES_MAX}

In [ ]:
# Merge everything into one YOLO dataset: classes stop_sign (+ do_not_enter if present)
!python train/prepare_dataset.py --inputs /kaggle/input {WORK}/open_images --out {WORK}/finetune

In [ ]:
# Fine-tune, export for the Pi, and compare against the pretrained model (~30-60 min on a T4)
!python train/train.py --data {WORK}/finetune/data.yaml --out {OUT} --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH}

In [ ]:
# Bundle the results into one zip for easy download from the Output tab
import shutil
shutil.copy(f"{WORK}/finetune/dataset_summary.json", OUT)
shutil.make_archive("/kaggle/working/stop_sign_finetune_results", "zip", OUT)
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/FINETUNE_REPORT.md").read()))
!ls -la {OUT}